# Multi-verifier attestation (replayable)

A claim hashes its subject, logic, **variance policy** and **consensus rule**. Code, models, agents and humans are actors (content-addressed). Each attestation is a Merkle leaf; the status (SUPPORTED / DISPUTED / FAILED / UNKNOWN) is recomputed from leaves + policy by anyone. Swap the claim / observations below.

In [1]:
import os, sys
from pathlib import Path
if Path.cwd().name == 'notebooks': os.chdir('..')
sys.path.insert(0, os.getcwd())
from baebugs import attest as A
code = A.actor('code', 'baebugs.insect_kg.rebuild_hash', 'main', key='main', self_or_nonself='self')
m1, m2 = A.actor('model', 'gemini-3-flash-preview'), A.actor('model', 'qwen3:1.7b')
c = A.claim({'toy': 'Apidae_mean_13.5mm'}, 'toy reconstructs to this plan and length', 'insect_lomo.check',
            {'plan': 'bee', 'length_mm': 13.5}, {'abs': 0.0, 'length_mm': {'abs': 0.1}},
            {'deterministic_required': True, 'min_agents': 2, 'quorum': 0.5})
leaves = [A.attest(c, code, {'plan': 'bee', 'length_mm': 13.5}), A.attest(c, m1, {'plan': 'bee', 'length_mm': 13.55}),
          A.attest(c, m2, {'plan': 'wasp', 'length_mm': 9.0})]
for l in leaves: print(l['actor_kind'], l['quadrant'], l['within_policy'], l['residual'])
print(A.claim_root(c, leaves))

code self/within True {'plan': None, 'length_mm': 0.0}
model non-self/within True {'plan': None, 'length_mm': 0.05}
model non-self/outside False {'plan': {'expected': 'bee', 'observed': 'wasp'}, 'length_mm': 4.5}
{'claim': '20b70ecc6306bb3473459067f7189f1a017919d4524107c3346fddc3fc9f048a', 'n': 3, 'root': 'ae672dd4b1f4a4a364053a20c49f20516b70ea7610574d506550799db179b314', 'status': 'SUPPORTED'}


## Information about information: attest the attestation

In [2]:
meta = A.claim({'attestation': leaves[2]['leaf']}, 'model leaf is outside policy', 'attest._within', False)
print(A.claim_root(meta, [A.attest(meta, code, leaves[2]['within_policy'])]))

{'claim': '34d0ad8a12c6357f7ba07ffa7f4f3bc73c12515cc54bcefc5f6e14a7763e2103', 'n': 1, 'root': 'eded08186af536aedab0b57ca4eb12f68da01f037d4d68c23550fabfabd728a3', 'status': 'SUPPORTED'}
